___
# <center>Probabilidade e incerteza</center>
___

## Aula 07

**Objetivo da aula:** ao final desta aula, você deve ser capaz de:

 * calcular a probabilidade de um evento como proporção na base;
 * calcular uma probabilidade condicional filtrando e recontando;
 * montar a tabela de duas variáveis e ler as contagens dela;
 * verificar se dois eventos são independentes comparando o observado com o que a independência preveria.

Este notebook é curto de propósito. Tudo que está aqui já foi feito na lousa
hoje: o que muda é que agora o computador faz a divisão, e você confere se o
número bate com o que você calculou à mão.


___
<div id="indice"></div>

## Índice

- [A base de hoje](#dados)

- [Probabilidade é uma proporção](#proporcao)

- [A tabela de dupla entrada](#tabela)

- [Probabilidade condicional](#condicional)

- [Independência: o teste](#independencia)

- [RESUMO](#resumo)


___
<div id="dados"></div>

# A base de hoje

A mesma base de apelações criminais das aulas 4 a 6. Uma linha por acórdão.


In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

URL = "https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados"

criminal = pd.read_csv(f"{URL}/tjsp_cjsg_criminal.csv")

# Fora as linhas sem regime: não dá para calcular proporção de regime
# em acórdão que não informou regime nenhum.
penas = criminal.dropna(subset=["regime_inicial"])

penas.shape


Duas colunas interessam hoje:

- `regime_inicial`: aberto, semiaberto ou fechado;
- `houve_reincidencia`: `True` ou `False`.


In [ ]:
penas[["regime_inicial", "houve_reincidencia"]].head()


[Volta ao Índice](#indice)


___
<div id="proporcao"></div>

# Probabilidade é uma proporção

$P(A)$ é o número de casos em $A$ dividido pelo total. No pandas, uma coluna de
`True` e `False` já sabe fazer essa conta sozinha: a média de uma coluna
booleana **é** a proporção de `True`.


In [ ]:
# O evento "o regime foi fechado", caso a caso
fechado = penas["regime_inicial"] == "fechado"

fechado.head()


In [ ]:
# P(fechado): quantos True, dividido pelo total
fechado.mean()


Por que a média funciona: `True` vale 1 e `False` vale 0, então somar a coluna
conta os casos e dividir pelo tamanho dá a proporção. É a definição de
probabilidade que usamos na lousa, escrita em uma linha.


**✍️ Agora você.** Calcule $P(\text{réu reincidente})$. É a mesma ideia, em outra coluna.


In [ ]:
reincidente = penas["________"]

reincidente.________()


[Volta ao Índice](#indice)


___
<div id="tabela"></div>

# A tabela de dupla entrada

`pd.crosstab` cruza duas colunas e conta quantos casos caem em cada
combinação. É exatamente a tabela que estava no slide.

Vale reter os dois nomes, porque eles aparecem juntos o tempo todo:

- o **miolo** da tabela é a distribuição **conjunta**, e cada célula responde
  por duas variáveis ao mesmo tempo;
- a última linha e a última coluna são as distribuições **marginais**, e cada
  valor delas responde por uma variável só.


✔️ **Uso do `pd.crosstab`**

```python
# Sintaxe geral:
pd.crosstab(coluna_das_linhas, coluna_das_colunas, margins=True)
```

Documentação oficial: [pd.crosstab](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)


In [ ]:
pd.crosstab(
    penas["regime_inicial"],
    penas["houve_reincidencia"],
    margins=True,
    margins_name="total",
)


Todos os números da aula estão aí: 83 é a conjunta de fechado com reincidente,
135 é a marginal dos reincidentes e 333 é o total.


[Volta ao Índice](#indice)


___
<div id="condicional"></div>

# Probabilidade condicional

Condicionar é trocar o denominador: em vez de dividir pelos 333, dividimos só
pelo grupo que interessa. No pandas isso é **filtrar e recontar**.


In [ ]:
# Só os reincidentes. Depois do filtro, o total já é outro.
so_reincidentes = penas.query("houve_reincidencia == True")

len(so_reincidentes)


In [ ]:
# P(fechado | reincidente): a mesma média de antes, dentro do filtro
(so_reincidentes["regime_inicial"] == "fechado").mean()


Compare com $P(\text{fechado}) = 0{,}456$ na base inteira. Saber que o réu é
reincidente mudou o número, e é isso que significa dizer que as duas variáveis
têm relação.


**✍️ Agora você.** Agora o outro lado: $P(\text{fechado} \mid \text{NÃO reincidente})$. Troque o filtro e refaça a média.


In [ ]:
so_primarios = penas.query("houve_reincidencia == ________")

(so_primarios["regime_inicial"] == "________").mean()


✔️ **Um atalho.** `normalize="columns"` faz o crosstab dividir cada coluna pelo
próprio total, que é a mesma conta condicional de uma vez só.


In [ ]:
pd.crosstab(
    penas["regime_inicial"],
    penas["houve_reincidencia"],
    normalize="columns",
).round(3)


⚠️ **Cuidado com o `normalize`.** `"columns"` divide por coluna, `"index"`
divide por linha e `"all"` divide pelo total geral. As três dão números
diferentes e respondem a perguntas diferentes. Errar aqui é o mesmo erro de
dividir por 333 em vez de 135.


[Volta ao Índice](#indice)


___
<div id="independencia"></div>

# Independência: o teste

Se dois eventos fossem independentes, a probabilidade dos dois juntos seria o
produto das duas. O teste é comparar esse produto com o que a base tem de
verdade.


In [ ]:
p_fechado = fechado.mean()
p_reincidente = reincidente.mean()

# O que a independência preveria, em número de acórdãos
esperado = p_fechado * p_reincidente * len(penas)

round(esperado, 1)


In [ ]:
# O que a base tem de verdade
observado = (fechado & reincidente).sum()

observado


Cerca de 62 contra 83. A diferença é o tamanho da relação entre reincidência e
regime fechado: 21 acórdãos que a independência não explica.

Na aula 17 vamos aprender a decidir se uma diferença dessas é grande o
bastante para não ser acaso. Por enquanto basta saber olhar para ela.


<div id="ex1"></div>

### EXERCÍCIO 1

Refaça a comparação para o par **regime aberto** e **reincidência**.

1. calcule $P(\text{aberto})$ e $P(\text{aberto} \mid \text{reincidente})$;
2. calcule quantos acórdãos a independência preveria e compare com o observado;
3. em uma frase: a relação vai na mesma direção da de regime fechado, ou na
   direção contrária?


In [ ]:
aberto = penas["regime_inicial"] == "________"

print("P(aberto)              =", round(aberto.________(), 3))
print("P(aberto | reincidente) =",
      round((so_reincidentes["regime_inicial"] == "________").mean(), 3))
print("esperado sob independência =",
      round(aberto.mean() * ________ * len(penas), 1))
print("observado                  =", (aberto & ________).sum())


[Volta ao Índice](#indice)


___
<div id="resumo"></div>

# RESUMO

| ideia | no pandas |
|---|---|
| $P(A)$ | média de uma coluna booleana |
| tabela de dupla entrada (conjunta e marginais) | `pd.crosstab(a, b, margins=True)` |
| $P(A \mid B)$ | `.query()` no B, e a média de A dentro do filtro |
| todas as condicionais de uma vez | `pd.crosstab(a, b, normalize="columns")` |
| independência | comparar $P(A) \times P(B) \times n$ com o observado |

**A frase para levar:** condicionar é trocar o denominador. Todo erro de
probabilidade que vimos hoje, inclusive os que prenderam gente inocente, é
alguma versão de dividir pelo número errado.


[Volta ao Índice](#indice)
